# Nsight Graphics GPU Trace — TSV Bundle Loader

Loads the TSV bundle produced by `ngfx --auto-export` into pandas.
Small files load fully; large files (e.g. the REGIMES-style raw sample
table, which can be 300+MB) are streamed/summarized first so you can see
the real column names before pulling a filtered subset into memory.

In [10]:
import csv
from pathlib import Path

import pandas as pd

BUNDLE_DIR = Path("../data")          # <-- point this at your --output-dir
LARGE_FILE_THRESHOLD_MB = 20
CHUNK_ROWS = 250_000

In [11]:
def sniff_delimiter(path: Path, sample_bytes: int = 8192) -> str:
    """Nsight exports are tab-separated even when the extension says .csv/.xls."""
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        sample = f.read(sample_bytes)
    try:
        return csv.Sniffer().sniff(sample, delimiters="\t,;").delimiter
    except csv.Error:
        return "\t"


def load_small_file(path: Path) -> pd.DataFrame:
    delim = sniff_delimiter(path)
    return pd.read_csv(path, sep=delim, engine="python", on_bad_lines="warn")


def summarize_large_file(path: Path, delim: str):
    """Stream through once to report shape/columns without loading it all."""
    n_rows, columns = 0, None
    for chunk in pd.read_csv(path, sep=delim, chunksize=CHUNK_ROWS,
                              low_memory=False, on_bad_lines="warn"):
        if columns is None:
            columns = list(chunk.columns)
        n_rows += len(chunk)
    print(f"    -> {n_rows:,} rows, {len(columns)} columns")
    print(f"    -> columns: {columns}")


def load_large_file_filtered(path: Path, delim: str, filter_fn=None, usecols=None) -> pd.DataFrame:
    """Stream a big file in chunks, optionally filtering rows / subsetting
    columns as you go, then concatenate. Use this for the REGIMES-style file
    once you know which columns you need — avoids loading 300+MB at once."""
    pieces = []
    for chunk in pd.read_csv(path, sep=delim, chunksize=CHUNK_ROWS,
                              usecols=usecols, low_memory=False,
                              on_bad_lines="warn"):
        if filter_fn is not None:
            chunk = chunk[filter_fn(chunk)]
        pieces.append(chunk)
    return pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame()

## Discover + load everything in the bundle

In [12]:
candidates = sorted(
    p for p in BUNDLE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {".tsv", ".csv", ".xls", ".txt"}
)

tables = {}   # small files loaded fully, keyed by filename stem
large_files = {}  # large files: just paths + delimiter, load on demand below

print(f"Found {len(candidates)} file(s) in {BUNDLE_DIR}:\n")

for path in candidates:
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"[{path.name}] ({size_mb:.1f} MB)")

    if size_mb < LARGE_FILE_THRESHOLD_MB:
        df = load_small_file(path)
        tables[path.stem] = df
        print(f"    -> loaded fully: {df.shape[0]:,} rows x {df.shape[1]} cols")
        print(f"    -> columns: {list(df.columns)}")
    else:
        delim = sniff_delimiter(path)
        large_files[path.stem] = (path, delim)
        print("    -> large file, summarizing only (load selectively below)")
        summarize_large_file(path, delim)
    print()

Found 8 file(s) in ../data:

[D3DPERF_EVENTS.xls] (0.0 MB)
    -> loaded fully: 0 rows x 61 cols
    -> columns: ['event_text', 'time_ms', 'time_ms.1', 'time_ms.2', 'time_ms.3', 'time_ms.4', 'time_ms.5', 'time_ms.6', 'time_ms.7', 'time_ms.8', 'time_ms.9', 'time_ms.10', 'time_ms.11', 'time_ms.12', 'time_ms.13', 'time_ms.14', 'time_ms.15', 'time_ms.16', 'time_ms.17', 'time_ms.18', 'time_ms.19', 'time_ms.20', 'time_ms.21', 'time_ms.22', 'time_ms.23', 'time_ms.24', 'time_ms.25', 'time_ms.26', 'time_ms.27', 'time_ms.28', 'time_ms.29', 'time_ms.30', 'time_ms.31', 'time_ms.32', 'time_ms.33', 'time_ms.34', 'time_ms.35', 'time_ms.36', 'time_ms.37', 'time_ms.38', 'time_ms.39', 'time_ms.40', 'time_ms.41', 'time_ms.42', 'time_ms.43', 'time_ms.44', 'time_ms.45', 'time_ms.46', 'time_ms.47', 'time_ms.48', 'time_ms.49', 'time_ms.50', 'time_ms.51', 'time_ms.52', 'time_ms.53', 'time_ms.54', 'time_ms.55', 'time_ms.56', 'time_ms.57', 'time_ms.58', 'time_ms.59']

[FRAME.xls] (0.0 MB)
    -> loaded fully: 0

## Inspect the small tables

`tables` now holds every small/metadata file as a DataFrame, keyed by filename stem.

In [13]:
for name, df in tables.items():
    print(name, df.shape)
    display(df.head())

D3DPERF_EVENTS (0, 61)


,event_text,time_ms,time_ms.1,time_ms.2,time_ms.3,time_ms.4,time_ms.5,time_ms.6,time_ms.7,time_ms.8,...,time_ms.50,time_ms.51,time_ms.52,time_ms.53,time_ms.54,time_ms.55,time_ms.56,time_ms.57,time_ms.58,time_ms.59


FRAME (0, 61)


,GPU frame time,18.51,18.2477,18.226,17.7527,17.9524,17.6826,17.9902,17.67,17.9782,...,18.8238,18.2277,18.4543,18.1001,18.251,18.3489.1,18.6262,17.7527.1,17.9566,17.7654


GPUTRACE_FRAME (150, 61)


...

,GPU Trace Data | BASE


activethreadsperwarp (1, 3)


,Active Threads Per Warp,Threads/Warp,Coherence
0,Active Threads Per Warp,29.1,90.9%


meshdxexample-capture-eventlist (4741, 5)


,Event,Description,Object,CPU Duration (ns),GPU Duration (ns)
0,0 - 78,Frame 49679,NaN,1344000,17667168
1,0,// Capture Begin,--,0,0
2,1,"vkWaitForFences(VkDevice device = 'Device_5', ...",Device_5,5000,0
3,2,"vkResetFences(VkDevice device = 'Device_5', ui...",Device_5,4000,0
4,3,vkAcquireNextImageKHR(VkDevice device = 'Devic...,Device_5,23000,0


## Pull a filtered slice from a large file

Once you've seen the real column names printed above, swap them in here.
This streams the file in chunks rather than loading the whole thing.

In [14]:
# Example — adjust the key and column names to match your actual bundle
big_name = next(iter(large_files))  # or set explicitly, e.g. "GPUTRACE_REGIMES"
path, delim = large_files[big_name]

df_big = load_large_file_filtered(
    path, delim,
    # usecols=["Frame", "Stage", "Duration (ns)"],       # <-- real column names
    # filter_fn=lambda c: c["Frame"].between(0, 59),      # <-- your 60-frame window
)

df_big.shape

StopIteration: 